# Basis-set sensitivity at 6e6o

Compare UCCSD in STO-3G and cc-pVDZ at the same active-space size. Each energy error and overlap is measured against CASCI **within the same basis**; equal active-space sizes do not imply identical orbitals. Basis comparisons pair molecules by name.

## Inputs and coverage

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import orth

from notebook_utils.operators import get_excitation_percentages
from config import PROCESSED_DATAFRAMES_DIR, WAVEFUNCTIONS_DIR
from notebook_utils.chem import CHEMICAL_ACCURACY
from notebook_utils.graphs import show_scrollable_figs

BASES = ("sto-3g", "cc-pVDZ")
STATES = ("s0", "s1", "t1", "t2")
SPECIAL = ["Butadiene", "Cyclopentadiene", "Furan", "Norbornadiene", "Pyrrole"]
CUTOFF = 1e-8


def load_rows(filename, expansions):
    path = PROCESSED_DATAFRAMES_DIR / filename
    if not path.exists():
        print(f"Missing: {path.name}")
        return pd.DataFrame(columns=["basis", "molecule", "expansion"])
    df = pd.read_pickle(path).query("ansatz == 'UCCSD' and active_space == '6e6o'").copy()
    if "basis" not in df and "basis_set" in df:
        df = df.rename(columns={"basis_set": "basis"})
    if "basis" not in df:
        raise ValueError(f"{filename} has no basis label. Export a basis column from notebook 00.")
    df["basis"] = df["basis"].str.lower().map({b.lower(): b for b in BASES})
    df = df[df.basis.notna() & df.expansion.isin(expansions)]
    return df.set_index(["basis", "molecule", "expansion"], verify_integrity=True).reset_index()


singles = load_rows("sv_qse_data.pkl", ["singlet", "triplet"])
doubles = load_rows("sv_qse_ov_doubles_data.pkl", ["select_ov_doubles"])
coverage = pd.concat([singles, doubles]).groupby(["basis", "expansion"]).size()
display(coverage.unstack(fill_value=0).reindex(index=BASES, columns=["singlet", "triplet", "select_ov_doubles"], fill_value=0))

## Index-matched low-state errors

Compare S0, S1 and the first two triplet energies (means of consecutive triplets of jointly diagonalised roots). The dotted guides mark chemical accuracy; the diagonal denotes equal error.

In [ ]:
def solve_row(row):
    values, vectors = np.linalg.eigh(row.S)
    keep = values > CUTOFF
    transform = vectors[:, keep] / np.sqrt(values[keep])
    energies, coefficients = np.linalg.eigh(transform.conj().T @ row.H @ transform)
    return energies, transform @ coefficients


def energy_error(energies, row, root):
    group = energies[3*root:3*root+3] if row.expansion == "triplet" else energies[root:root+1]
    if len(group) != (3 if row.expansion == "triplet" else 1):
        return np.nan
    return 1000 * abs(group.mean() - row.pyscf_casci_energies[root])


records, q1_vectors = [], {}
for row in singles.itertuples():
    energies, coefficients = solve_row(row)
    labels = STATES[:2] if row.expansion == "singlet" else STATES[2:]
    for root, state in enumerate(labels):
        records.append((row.molecule, row.basis, state, energy_error(energies, row, root)))
    if row.expansion == "singlet" and coefficients.shape[1] > 1 and row.molecule in SPECIAL:
        q1_vectors[row.basis, row.molecule] = coefficients[:, 1]

errors = pd.DataFrame(records, columns=["molecule", "basis", "state", "error_mHa"])

In [ ]:
def paired_panels(data, metric, floor, guide=None, highlight=False):
    fig, axes = plt.subplots(2, 2, figsize=(9, 8), constrained_layout=True)
    summaries = []
    for state, ax in zip(STATES, axes.flat):
        wide = data[data.state == state].pivot(index="molecule", columns="basis", values=metric)
        pairs = wide.reindex(columns=BASES).dropna()
        if not pairs.empty:
            colours = ["tab:orange" if highlight and state == "s1" and m in SPECIAL else "tab:blue" for m in pairs.index]
            ax.scatter(*pairs.clip(lower=floor).to_numpy().T, c=colours)
            lo, hi = pairs.clip(lower=floor).to_numpy().min(), pairs.clip(lower=floor).to_numpy().max()
            ax.plot([lo/2, hi*2], [lo/2, hi*2], "--", color="0.5")
            ax.set(xscale="log", yscale="log")
            if guide is not None:
                ax.axvline(guide, ls=":", color="0.5")
                ax.axhline(guide, ls=":", color="0.5")
        else:
            ax.text(.5, .5, "No paired molecules yet", transform=ax.transAxes, ha="center")
        ax.set(title=f"{state.upper()} (n={len(pairs)})", xlabel="STO-3G", ylabel="cc-pVDZ")
        for basis in BASES:
            v = pairs[basis]
            stats = dict(state=state, basis=basis, n=len(v), median=v.median(), IQR=v.quantile(.75) - v.quantile(.25))
            if guide is not None:
                stats["fraction_within_chemical_accuracy"] = (v <= guide).mean()
            summaries.append(stats)
    fig.suptitle(metric)
    display(pd.DataFrame(summaries))
    show_scrollable_figs([fig])


paired_panels(errors, "error_mHa", 1e-10, 1000 * CHEMICAL_ACCURACY)

In [ ]:
# Compare all four errors on the same molecules in both bases.
all_states = errors.pivot(index="molecule", columns=["basis", "state"], values="error_mHa")
all_states = all_states.reindex(columns=pd.MultiIndex.from_product([BASES, STATES])).dropna()
hierarchy = []
for basis in BASES:
    values = all_states[basis].to_numpy(dtype=float)
    largest = np.isclose(values, values.max(axis=1, keepdims=True), rtol=0, atol=1e-8)
    s1_largest = largest[:, 1]
    tied = largest.sum(axis=1) > 1
    hierarchy.append((basis, len(values), (s1_largest & ~tied).sum(), (s1_largest & tied).sum()))
display(pd.DataFrame(hierarchy, columns=["basis", "n", "S1_unique_largest", "S1_tied_largest"]))

## Exact-state projection onto the singles QSE basis

Use the full column span of the determinant-space basis matrix, independently of the energy solver cutoff. Triplet projections average the three exact spin sectors. Orange S1 points identify the five selected molecules. Deficits below 1e-7% are floored only for plotting.

In [ ]:
def unit(vector):
    return vector / np.linalg.norm(vector)


records, overlaps = [], []
for row in singles.itertuples():
    path = WAVEFUNCTIONS_DIR / "qse_state_matrices" / row.basis / "6e6o" / row.expansion / f"{row.molecule}_UCCSD.npz"
    if not path.exists():
        continue
    with np.load(path) as archive:
        matrix = archive["qse_state_matrix"]
    span = orth(matrix)
    exact = row.pyscf_casci_pvec
    labels = STATES[:2] if row.expansion == "singlet" else STATES[2:]
    for root, state in enumerate(labels):
        keys = [str(root)] if row.expansion == "singlet" else [f"{root}_{s}" for s in ("0", "p1", "m1")]
        projection = np.mean([np.linalg.norm(span.conj().T @ unit(exact[k]))**2 for k in keys])
        records.append((row.molecule, row.basis, state, projection, 100 * (1 - projection)))
    key = (row.basis, row.molecule)
    if row.expansion == "singlet" and key in q1_vectors:
        q1 = unit(matrix @ q1_vectors[key])
        overlaps.append((*key, *[100 * abs(np.vdot(unit(exact[k]), q1))**2 for k in ("1", "2")]))

projections = pd.DataFrame(records, columns=["molecule", "basis", "state", "projection", "deficit_percent"])
paired_panels(projections, "deficit_percent", 1e-7, highlight=True)

### Does S1 projection deficit track its energy error?

Compare the same molecules on shared axes. Dotted guides denote 1% projection deficit and chemical accuracy.

In [ ]:
s1 = projections.query("state == 's1'").merge(errors, on=["basis", "molecule", "state"], validate="one_to_one")
s1 = s1.dropna(subset=["deficit_percent", "error_mHa"])
s1 = s1[s1.groupby("molecule").basis.transform("nunique") == len(BASES)]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True, constrained_layout=True)
for basis, ax in zip(BASES, axes):
    points = s1[s1.basis == basis]
    ax.scatter(points.deficit_percent.clip(lower=1e-7), points.error_mHa.clip(lower=1e-10))
    ax.axvline(1, color="0.4", ls=":")
    ax.axhline(1000 * CHEMICAL_ACCURACY, color="0.4", ls=":")
    ax.set(xscale="log", yscale="log", title=f"{basis} (n={len(points)})", xlabel="S1 projection deficit (%)")
    if points.empty:
        ax.text(.5, .65, "No paired molecules yet", ha="center", transform=ax.transAxes)
axes[0].set_ylabel("Absolute S1 energy error (mHa)")
show_scrollable_figs([fig])

## Does S1 have more exact double-excitation weight than T1 and T2?

Compare mean CASCI doubles percentages relative to each basis set's closed-shell HF determinant. Triplet weights average the three sectors. Differences are in percentage points; S0 is included as a ground-state comparison.

In [ ]:
hf_state = "11" * 3 + "00" * 3
weights = []
for row in singles.itertuples():
    labels = STATES[:2] if row.expansion == "singlet" else STATES[2:]
    for root, state in enumerate(labels):
        keys = [str(root)] if row.expansion == "singlet" else [f"{root}_{s}" for s in ("0", "p1", "m1")]
        sectors = [get_excitation_percentages(row.pyscf_casci_pvec[k], hf_state)[2] for k in keys]
        weights.append((row.molecule, row.basis, state, np.mean(sectors)))

double_weights = pd.DataFrame(weights, columns=["molecule", "basis", "state", "weight_percent"])
paired_weights = double_weights.pivot(index="molecule", columns=["basis", "state"], values="weight_percent")
paired_weights = paired_weights.reindex(columns=all_states.columns).dropna()
weight_summary = paired_weights.mean().unstack().reindex(index=BASES, columns=STATES)
weight_summary.insert(0, "n", len(paired_weights))
for triplet in ("t1", "t2"):
    weight_summary[f"s1_minus_{triplet}_pp"] = weight_summary.s1 - weight_summary[triplet]
display(weight_summary)

## Effect of adding OV doubles on S1

Each point compares operator pools for one molecule within one basis. Positive reductions denote smaller absolute errors with OV doubles.

In [ ]:
ov_records = []
for row in doubles.itertuples():
    energies, _ = solve_row(row)
    ov_records.append((row.molecule, row.basis, energy_error(energies, row, 1)))

ov = pd.DataFrame(ov_records, columns=["molecule", "basis", "ov_mHa"])
comparison = errors.query("state == 's1'").merge(ov, on=["molecule", "basis"], validate="one_to_one").dropna()
comparison["reduction_mHa"] = comparison.error_mHa - comparison.ov_mHa
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
summary = []
for basis, ax in zip(BASES, axes):
    pairs = comparison[comparison.basis == basis]
    summary.append((basis, len(pairs), pairs.error_mHa.median(), pairs.ov_mHa.median(),
                    pairs.reduction_mHa.median(), (pairs.reduction_mHa > 0).sum()))
    if not pairs.empty:
        x, y = pairs[["error_mHa", "ov_mHa"]].clip(lower=1e-10).to_numpy().T
        ax.scatter(x, y, color="black")
        limits = [min(x.min(), y.min())/2, max(x.max(), y.max())*2]
        ax.plot(limits, limits, ":", color="0.4")
        ax.set(xscale="log", yscale="log", xlim=limits, ylim=limits)
    else:
        ax.text(.5, .5, "No paired molecules yet", ha="center", transform=ax.transAxes)
    ax.set(title=f"{basis} (n={len(pairs)})", xlabel="Singles S1 error (mHa)", ylabel="OV doubles S1 error (mHa)")
display(pd.DataFrame(summary, columns=["basis", "n", "median_singles_mHa", "median_OV_mHa", "median_reduction_mHa", "improved"]))
show_scrollable_figs([fig])

## Direct Q1 overlaps for the five selected molecules

Squared overlap (%) with each exact singlet, without root assignment. Blank entries indicate unavailable inputs.

These energy indices are local to each basis. A change in the larger overlap does not establish a cross-basis state crossing or its orbital cause.

In [ ]:
q1_overlaps = pd.DataFrame(overlaps, columns=["basis", "molecule", "S1", "S2"])
display(q1_overlaps.set_index(["molecule", "basis"]).reindex(pd.MultiIndex.from_product(
    [SPECIAL, BASES], names=["molecule", "basis"])))